## 7. Elo_Ranking

Calcula un rating Elo por club, partido a partido, a partir de `silver.matches_transformed`
y escribe `gold.club_elo` (una fila por club y partido, con el Elo antes y despues).

- Cada liga es un universo propio: los clubes solo se enfrentan dentro de su liga.
- Todos arrancan en 1500 en la primera temporada; los ascendidos entran con 1450.
- Al inicio de cada temporada los ratings regresan un 25 % hacia 1500.
- Ventaja de local de 60 puntos y K = 20, multiplicado por la diferencia de goles.

In [0]:
dbutils.widgets.removeAll()

In [0]:
import pandas as pd
from pyspark.sql.functions import col

In [0]:
dbutils.widgets.text("catalogo", "football_dev")
dbutils.widgets.text("esquema_source", "silver")
dbutils.widgets.text("esquema_sink", "gold")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
ELO_INICIAL = 1500
ELO_ASCENDIDO = 1450
K = 20
VENTAJA_LOCAL = 60
REGRESION_TEMPORADA = 0.25

In [0]:
def multiplicador_goles(diferencia):
    diferencia = abs(diferencia)
    if diferencia <= 1:
        return 1.0
    if diferencia == 2:
        return 1.5
    return (11 + diferencia) / 8


def calcular_elo(partidos):
    filas = []
    ordenados = partidos.sort_values(["match_date", "match_id"])
    for (liga, pais), df_liga in ordenados.groupby(["league_name", "country"], sort=False):
        ratings = {}
        temporada_actual = None
        primera_temporada = True
        for p in df_liga.itertuples(index=False):
            if p.season_year != temporada_actual:
                primera_temporada = temporada_actual is None
                temporada_actual = p.season_year
                ratings = {c: r + REGRESION_TEMPORADA * (ELO_INICIAL - r) for c, r in ratings.items()}
            for club in (p.home_club, p.away_club):
                if club not in ratings:
                    ratings[club] = ELO_INICIAL if primera_temporada else ELO_ASCENDIDO

            elo_local, elo_visita = ratings[p.home_club], ratings[p.away_club]
            esperado_local = 1 / (1 + 10 ** ((elo_visita - elo_local - VENTAJA_LOCAL) / 400))
            if p.home_goals > p.away_goals:
                real_local, res_local, res_visita = 1.0, "W", "L"
            elif p.home_goals < p.away_goals:
                real_local, res_local, res_visita = 0.0, "L", "W"
            else:
                real_local, res_local, res_visita = 0.5, "D", "D"
            cambio = K * multiplicador_goles(p.home_goals - p.away_goals) * (real_local - esperado_local)
            ratings[p.home_club] = elo_local + cambio
            ratings[p.away_club] = elo_visita - cambio

            comun = {"match_id": p.match_id, "match_date": p.match_date, "season_year": p.season_year,
                     "round": p.round, "country": pais, "league_name": liga}
            filas.append({**comun, "club": p.home_club, "opponent": p.away_club, "is_home": True,
                          "goals_for": p.home_goals, "goals_against": p.away_goals, "result": res_local,
                          "elo_pre": elo_local, "opponent_elo_pre": elo_visita,
                          "elo_post": elo_local + cambio, "elo_change": cambio})
            filas.append({**comun, "club": p.away_club, "opponent": p.home_club, "is_home": False,
                          "goals_for": p.away_goals, "goals_against": p.home_goals, "result": res_visita,
                          "elo_pre": elo_visita, "opponent_elo_pre": elo_local,
                          "elo_post": elo_visita - cambio, "elo_change": -cambio})
    return pd.DataFrame(filas)

In [0]:
df_partidos = spark.table(f"{catalogo}.{esquema_source}.matches_transformed").select(
    "match_id", "match_date", "season_year", "round", "country", "league_name",
    "home_club", "away_club", "home_goals", "away_goals"
).toPandas()

df_elo = calcular_elo(df_partidos)

In [0]:
TIPOS_ELO = [
    ("match_id", "int"), ("match_date", "date"), ("season_year", "int"), ("round", "int"),
    ("country", "string"), ("league_name", "string"), ("club", "string"), ("opponent", "string"),
    ("is_home", "boolean"), ("goals_for", "int"), ("goals_against", "int"), ("result", "string"),
    ("elo_pre", "double"), ("opponent_elo_pre", "double"), ("elo_post", "double"), ("elo_change", "double"),
]

df_gold_elo = spark.createDataFrame(df_elo).select([col(c).cast(t).alias(c) for c, t in TIPOS_ELO])
df_gold_elo.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.club_elo")